In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
block_size = 3 # context length: how many characters do we take to predict the next one?
  
X, Y = [], []
for w in words[:5]:

    #print(w)
    context = [0] * block_size
    for ch in w + '.': # always padding with dots
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)
print(X.shape, Y.shape)


... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
torch.Size([32, 3]) torch.Size([32])


In [6]:
X.shape, X.dtype, Y.shape,Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
C=torch.randn((27,2))


In [8]:
emb=C[X]
emb.shape

torch.Size([32, 3, 2])

In [9]:
# F.one_hot(torch.tensor(5),num_classes=27).float() @ C
# must be tensor not an int
# embedding of the integer can be thought of as either indexing into a lookup table C or as a first layer(linear) of a bigger neural net and their weight matrix is C and encoding integers in one hot and the first layer embeds them
# using index here because its faster

In [10]:
# C[torch.tensor([5,6,7,7,6])]
C[X].shape

torch.Size([32, 3, 2])

In [12]:
W1=torch.randn((3*2,100)) # 6= 3*2 which are the other two dimensions of emb
b1=torch.randn(100)

In [ ]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape # need to specify index mannualy

torch.Size([32, 6])

In [18]:
torch.unbind(emb,1) #gives list of tensors removing index 1

(tensor([[-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.5856,  1.7536],
         [-0.9739, -0.0032],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [ 0.3087,  1.1016],
         [-1.6338, -0.8928],
         [-0.6752, -0.7332],
         [-0.9776,  0.7699],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [ 0.8502,  1.3997],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.6752, -0.7332],
         [ 1.1064, -1.4837],
         [ 0.8502,  1.3997],
         [-0.0027,  0.1607],
         [-0.5856,  1.7536],
         [-1.6338, -0.8928],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [ 1.1064, -1.4837],
         [ 0.3087,  1.1016],
         [-0.5063,  1.0186],
         [ 1.0765,  1.3294]]),
 tensor([[-0.0769,  1.4733],
         [-0.0769,  1.4733],
         [-0

In [ ]:
# so to concatenate without mentioning index manually
torch.cat(torch.unbind(emb,1),1) # inefficient as it creates a new tensor which takes up extra memory. 
#got the same result

torch.Size([32, 6])

In [22]:
a=torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [ ]:
a.view(2,9) # change the interpretation of a without changing the storage or properties

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [ ]:
a.view(3,3,2) # 3*3*2=18

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [ ]:
a.storage() # always stored as 1d

C:\Users\Dhruv\AppData\Local\Temp\ipykernel_17308\214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [ ]:
emb.view(32,6) # easiest using torch internals

tensor([[-0.0769,  1.4733, -0.0769,  1.4733, -0.0769,  1.4733],
        [-0.0769,  1.4733, -0.0769,  1.4733, -0.5856,  1.7536],
        [-0.0769,  1.4733, -0.5856,  1.7536, -0.9739, -0.0032],
        [-0.5856,  1.7536, -0.9739, -0.0032, -0.9739, -0.0032],
        [-0.9739, -0.0032, -0.9739, -0.0032,  0.8502,  1.3997],
        [-0.0769,  1.4733, -0.0769,  1.4733, -0.0769,  1.4733],
        [-0.0769,  1.4733, -0.0769,  1.4733,  0.3087,  1.1016],
        [-0.0769,  1.4733,  0.3087,  1.1016, -1.6338, -0.8928],
        [ 0.3087,  1.1016, -1.6338, -0.8928, -0.6752, -0.7332],
        [-1.6338, -0.8928, -0.6752, -0.7332, -0.9776,  0.7699],
        [-0.6752, -0.7332, -0.9776,  0.7699, -0.6752, -0.7332],
        [-0.9776,  0.7699, -0.6752, -0.7332,  0.8502,  1.3997],
        [-0.0769,  1.4733, -0.0769,  1.4733, -0.0769,  1.4733],
        [-0.0769,  1.4733, -0.0769,  1.4733,  0.8502,  1.3997],
        [-0.0769,  1.4733,  0.8502,  1.3997, -0.9776,  0.7699],
        [ 0.8502,  1.3997, -0.9776,  0.7

In [ ]:
h=torch.tanh(emb.view(-1,6)@ W1 + b1) 
# 100 d activations of 32 examples
# -1 makes pytorch infer that number given the other numbers

In [28]:
h

tensor([[-0.6101, -0.9999, -0.4239,  ..., -0.9986, -0.9995,  0.9969],
        [-0.8763, -1.0000, -0.6690,  ..., -0.9999, -0.9990,  0.9974],
        [ 0.8504, -0.9784, -0.9906,  ..., -0.9997, -0.9998,  0.8998],
        ...,
        [-0.0675, -0.9926,  0.7520,  ..., -0.3172, -0.9994,  0.9953],
        [ 0.9550, -0.9880, -0.9678,  ..., -0.9981, -0.9831,  0.9828],
        [-0.6024, -0.9972,  0.6847,  ..., -0.9342, -0.6783,  0.9617]])

In [ ]:
b1.shape
# 32*100
# 1*100 copied 32 times that is same bias added to all rows which is what we want

torch.Size([100])

In [30]:
#output layer
W2 = torch.randn((100,27)) # previous layer , output alphabets 
b2=torch.randn(27)

In [31]:
logits =h @ W2 +b2

In [32]:
logits.shape

torch.Size([32, 27])

In [33]:
counts=logits.exp()
probs=counts/counts.sum(1,keepdim=True)

In [34]:
probs.shape

torch.Size([32, 27])

In [ ]:
probs[0].sum() # means normalised

tensor(1.0000)

In [38]:
loss=probs[torch.arange(32),Y].log().mean()
loss

tensor(-15.4889)

In [39]:
# cleaned version